In [1]:
from pathlib import Path
import numpy as np
import pyedflib
from scipy.io.wavfile import write

DATA = Path("/media/ensound/Ventoy/apnea_data")
SIGNAL_NAMES = ("pressure", "thermistor", "effort_tho", "effort_abd")

In [ ]:
import json
import matplotlib.pyplot as plt
from scipy.signal import resample_poly, hilbert
from scipy.stats import theilslopes

corr_time = 150 #s
max_lag = 2 #s


def measure_correlation(reader):
    labels = reader.getSignalLabels()
    mic = reader.readSignal(labels.index("Mic"), digital=True)
    nonzero = np.flatnonzero(mic)
    end_valid = (nonzero[-1] + 1) / 48000 if nonzero.size else 0.0

    times, correlations = [], []
    lags = np.arange(-max_lag*500, max_lag*500+1, 25)
    starts = np.unique(np.linspace(0, end_valid - corr_time, 4).astype(int)) if end_valid >= corr_time else []
    for start in starts:
        sn = reader.readSignal(labels.index("Snore"), start=start * 500, n=corr_time * 500, digital=True).astype(float)
        sn = np.abs(hilbert(sn))
        tr = reader.readSignal(labels.index("Tracheal"), start=start * 48000, n=corr_time * 48000, digital=True,).astype(float)
        tr = resample_poly(tr, 1, 96)
        tr = np.abs(hilbert(tr))

        x = tr[max_lag * 500:(corr_time - max_lag) * 500]
        x = x - x.mean()

        corr = []
        for lag in lags:
            y = sn[max_lag * 500 + lag:(corr_time-max_lag) * 500 + lag]
            y = y - y.mean()
            den = np.linalg.norm(x) * np.linalg.norm(y)
            corr.append(abs(x @ y) / den if den else 0.0)

        times.append(start + corr_time / 2)
        correlations.append(corr)

    return dict(times=np.array(times), lags=lags / 500,
                correlations=np.array(correlations).reshape(-1, len(lags)),
                duration=float(reader.file_duration), end_valid=end_valid)



def plot_patient_calibration(patient, points, good, fit, destination):
    x, y = points[:, 0], points[:, 1]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout="constrained")

    if fit is not None:
        slope, b, residual = fit
        line = np.array([x.min(), x.max()])
        ax1.plot(line / 3600, slope * line + b, color="black", label="Linear fit")
        ax2.scatter(x[good] / 3600, residual[good] * 1000, label="Kept")
        ax1.text(.02, .04, f"a={1+slope:.10f}, b={b:.4f} s; drift={slope*3600:.4f} s/hour",
                 transform=ax1.transAxes)

    ax1.scatter(x[good] / 3600, y[good], label="Kept")
    ax1.scatter(x[~good] / 3600, y[~good], color="red", marker="x", label="Outliers")
    ax1.set(ylabel="Delay PSG − audio (s)", title=f"{patient}: {good.sum()}/{len(x)} points")
    ax1.legend()

    ax2.axhline(0, color="gray")
    ax2.set(xlabel="Audio time (hours)", ylabel="Final fit residuals (ms)")

    for ax in (ax1, ax2):
        ax.grid(alpha=.2)

    fig.savefig(destination.with_suffix(".png"), dpi=150)
    plt.close(fig)


def calibrate_patient(segment_files, destination):
    patient = int(destination.stem.split("-")[0])
    points, offset = [], 0.0

    for path in segment_files:
        with np.load(path) as part:
            for t, corr in zip(part["times"], part["correlations"]):
                k = np.argmax(corr)
                points.append([float(t + offset), float(part["lags"][k]), float(corr[k])])
            offset += float(part["duration"])

    points = np.array(points).reshape(-1, 3)
    x, y = points[:, 0], points[:, 1]
    good = np.zeros(len(x), dtype=bool)
    result = dict(a=None, b=None)
    fit = None

    if len(x) >= 3 and np.ptp(x) > 0:
        slope, b, _, _ = theilslopes(y, x)
        good = np.abs(y - (slope * x + b)) <= .15

        if good.sum() >= 3 and np.ptp(x[good]) > 0:
            slope, b = np.polyfit(x[good], y[good], 1)
            residual = y - (slope * x + b)
            fit = (slope, b, residual)
            result.update(
                a=float(1 + slope),
                b=float(b),
                inliers=int(good.sum()),
                total=len(x),
                median_residual=float(np.median(abs(residual[good]))),
                max_residual=float(np.max(abs(residual[good])))
            )
    result.update(points=points.tolist(), inlier_mask=good.tolist())
    plot_patient_calibration(patient, points, good, fit, destination)
    destination.write_text(json.dumps(result, indent=2))
    import csv
    root = destination.parent.parent
    with (root / "affines.csv").open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["patient_id", "a", "b"])
        for path in sorted(root.glob("*/*.json")):
            saved = json.loads(path.read_text())
            if saved.get("a") is not None:
                writer.writerow([int(path.stem), saved["a"], saved["b"]])
    return result


In [3]:
import wave

def extract_streams(edf_file, wav_file, signals_file, correlation_file):
    with pyedflib.EdfReader(str(edf_file)) as reader:
        if not correlation_file.exists():
            correlation = measure_correlation(reader)
            np.savez_compressed(correlation_file, **correlation)
        labels = reader.getSignalLabels()
        if not wav_file.exists():
            mic = labels.index("Mic")
            temporary = wav_file.with_suffix(".wav.part")
            write(temporary, int(reader.getSampleFrequency(mic)),reader.readSignal(mic, digital=True).astype(np.int16))
            temporary.replace(wav_file)

        transducers = [reader.getTransducer(i).strip() for i in range(reader.signals_in_file)]
        channels = [next(i for i, label in enumerate(labels) if label == "Flow Patient" and transducers[i] == sensor) for sensor in ("Pressure", "Thermistor")]
        channels += [labels.index("Effort THO"), labels.index("Effort ABD")]
        temporary = signals_file.with_suffix(".npz.part")
        with temporary.open("wb") as f:
            np.savez_compressed(f, **{name: reader.readSignal(i, digital=True).astype(np.int16) for name, i in zip(SIGNAL_NAMES, channels)})
        temporary.replace(signals_file)

def save_audio_metadata(audio, correlation_folder):
    segments = []
    for path in sorted(audio.glob("*.wav")):
        with wave.open(str(path)) as reader:
            count = reader.getnframes()
        with np.load(correlation_folder / f"{path.stem}.correlation.npz") as data:
            end = round(float(data["end_valid"]) * 48000)
        assert 0 <= end <= count, f"Invalid audio support: {path}"
        segments.append(dict(file=path.name, count=count, end_valid=end))
    (audio / "metadata.json").write_text(json.dumps(dict(sample_rate=48000, segments=segments), indent=2))


In [ ]:
from urllib.request import urlopen
from urllib.parse import quote
import json
import shutil

REPO = "Samuelsantos777/psg-audio-v3-unofficial-mirror"

def listing(folder):
    url = f"https://huggingface.co/api/datasets/{REPO}/tree/main/{folder}?limit=1000"
    with urlopen(url, timeout=180) as response:
        return json.load(response)

def download(remote, local):
    url = f"https://huggingface.co/datasets/{REPO}/resolve/main/" + quote(remote, safe="/")
    temporary = local.with_name(local.name + ".part")
    with urlopen(url, timeout=180) as response, temporary.open("wb") as f:
        shutil.copyfileobj(response, f)
    temporary.replace(local)

patients = sorted(Path(e["path"]).name for e in listing("APNEA_EDF") if e["type"] == "directory")
(DATA / "annotations").mkdir(parents=True, exist_ok=True)
alignment = DATA / "alignment"
alignment.mkdir(exist_ok=True)
temp_edf = DATA / "temp.edf"

for remote_patient in patients:
    patient = str(int(remote_patient.split("-")[0]))
    annotation = DATA / "annotations" / f"{patient}.rml"
    if not annotation.exists():
        download(f"APNEA_RML/{remote_patient}.rml", annotation)

    audio = DATA / "audio" / patient
    signals = DATA / "signals" / patient
    audio.mkdir(parents=True, exist_ok=True)
    signals.mkdir(parents=True, exist_ok=True)
    correlation_folder = alignment / patient
    correlation_folder.mkdir(exist_ok=True)

    segment_files = []
    patient_ok = True
    for entry in sorted(listing(f"APNEA_EDF/{remote_patient}"), key=lambda e: e["path"]):
        if not entry["path"].endswith(".edf"):
            continue
        stem = Path(entry["path"]).stem.replace(remote_patient, patient)
        wav_file = audio / f"{stem}.wav"
        signals_file = signals / f"{stem}.npz"
        correlation_file = correlation_folder / f"{stem}.correlation.npz"
        segment_files.append(correlation_file)
        if wav_file.exists() and signals_file.exists() and correlation_file.exists():
            continue
        print(f"Downloading: {patient}/{stem}", flush=True)
        download(entry["path"], temp_edf)
        try:
            extract_streams(temp_edf, wav_file, signals_file, correlation_file)
        except OSError as error:
            patient_ok = False
            print(f"Skipping {patient}: {error}")
            temp_edf.unlink(missing_ok=True)
            break
        temp_edf.unlink()
        print(f"Completed: {patient}/{stem}")

    if patient_ok:
        save_audio_metadata(audio, correlation_folder)
        result = calibrate_patient(segment_files, correlation_folder / f"{patient}.json")
        print(f"Alignment: {patient}: a={result['a']}, b={result['b']}")
    else:
        (correlation_folder / f"{patient}.json").write_text(json.dumps(dict(a=None, b=None)))
        (correlation_folder / f"{patient}.png").unlink(missing_ok=True)


Alignment: 00000995-100507: a=0.9999571629686247, b=0.20742742621990534
Alignment: 00000999-100507: a=0.9999551746390651, b=-0.19963350307968095
Alignment: 00001000-100507: a=0.9999569216268046, b=-0.6769233157946165
Alignment: 00001006-100507: a=0.9999567324492022, b=0.3467671782167632
Alignment: 00001008-100507: a=0.9999542479518556, b=-0.1688790478017458
Alignment: 00001010-100507: a=0.9999566433732765, b=0.26452895300487744
Alignment: 00001014-100507: a=0.9999568037188566, b=0.27459232726564425
Alignment: 00001016-100507: a=0.9999579062502931, b=0.22892969334879784
Alignment: 00001018-100507: a=0.9999562088601784, b=0.018472471056736092
Alignment: 00001020-100507: a=0.9999554254056254, b=0.10684002910296538
Downloading: 00001022-100507/00001022-100507[001]
